# 03 — Data Cleaning: Dataset v1 → v2

This notebook applies row-level filters and type casting to produce a clean, analysis-ready dataset.

**What this notebook does:**
- Drops `region = 'Global'` (derived aggregate, not a real country)
- Drops South Korea, Russia, Ukraine, Luxembourg, Egypt, Morocco (<50% temporal coverage)
- Drops rows with NULL/empty `track_id`
- Removes exact duplicate rows
- Casts all 28 source columns from VARCHAR to proper types
- Exports cleaned data as Hive-partitioned Parquet to `datasets/processed/v2/full/`

**What this notebook does NOT do** (deferred to feature engineering):
- No outlier removal
- No cultural distance imputation
- No `available_markets` parsing

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path
import tempfile

PROJECT_ROOT = Path.cwd().parent
V1_PATH = PROJECT_ROOT / "datasets" / "processed" / "v1" / "full"
V2_PATH = PROJECT_ROOT / "datasets" / "processed" / "v2" / "full"
EXCLUDED_REGIONS = ('South Korea', 'Russia', 'Ukraine', 'Luxembourg', 'Egypt', 'Morocco')

# Use disk-backed DuckDB to handle 26M+ rows without OOM
DB_PATH = Path(tempfile.mktemp(suffix=".duckdb"))
con = duckdb.connect(database=str(DB_PATH))
con.execute("SET preserve_insertion_order=false")

con.execute(f"""
    CREATE OR REPLACE VIEW spotify_v1 AS
    SELECT * FROM read_parquet('{V1_PATH}/year=*/**.parquet', hive_partitioning=true)
""")
print("View spotify_v1 created.")
print(f"V1 path: {V1_PATH}")
print(f"V2 path: {V2_PATH}")
print(f"Temp DB: {DB_PATH}")

View spotify_v1 created.
V1 path: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v1/full
V2 path: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v2/full
Temp DB: /var/folders/bl/m0zs7txj4lg0kmvg2zzypg840000gn/T/tmp4tmpx0m6.duckdb


## 2. Pre-Cleaning Audit

In [28]:
# Row count and schema overview
row_count = con.execute("SELECT COUNT(*) FROM spotify_v1").fetchone()[0]
schema = con.execute("DESCRIBE spotify_v1").fetchdf()

varchar_count = (schema["column_type"] == "VARCHAR").sum()
typed_count = len(schema) - varchar_count

print(f"Total rows: {row_count:,}")
print(f"Total columns: {len(schema)}")
print(f"VARCHAR columns: {varchar_count}")
print(f"Already typed columns: {typed_count}")
print()
schema

CatalogException: Catalog Error: Table with name spotify_v1 does not exist!
Did you mean "spotify_v2_disk"?

LINE 1: SELECT COUNT(*) FROM spotify_v1
                             ^

In [3]:
# Null rates for key columns
con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN track_id IS NULL OR track_id = '' THEN 1 ELSE 0 END) AS null_track_id,
        SUM(CASE WHEN streams IS NULL OR streams = '' THEN 1 ELSE 0 END) AS null_streams,
        SUM(CASE WHEN date IS NULL OR date = '' THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN rank IS NULL OR rank = '' THEN 1 ELSE 0 END) AS null_rank,
        SUM(CASE WHEN release_date IS NULL OR release_date = '' THEN 1 ELSE 0 END) AS null_release_date
    FROM spotify_v1
""").fetchdf()

,total_rows,null_track_id,null_streams,null_date,null_rank,null_release_date
0,26174269,304251.0,5852308.0,0.0,0.0,304251.0


In [4]:
# Count Global region rows
global_count = con.execute("""
    SELECT COUNT(*) FROM spotify_v1 WHERE region = 'Global'
""").fetchone()[0]
print(f"Rows with region = 'Global': {global_count:,} ({global_count/row_count*100:.2f}%)")

Rows with region = 'Global': 451,804 (1.73%)


In [5]:
# Country temporal coverage analysis
coverage = con.execute("""
    SELECT
        region,
        MIN(TRY_CAST(date AS DATE)) AS first_date,
        MAX(TRY_CAST(date AS DATE)) AS last_date,
        COUNT(DISTINCT TRY_CAST(date AS DATE)) AS active_days,
        ROUND(COUNT(DISTINCT TRY_CAST(date AS DATE)) * 100.0 / 1826, 1) AS coverage_pct
    FROM spotify_v1
    WHERE region != 'Global'
    GROUP BY region
    ORDER BY coverage_pct ASC
""").fetchdf()

print("Countries below 50% coverage threshold:")
print(coverage[coverage["coverage_pct"] < 50].to_string(index=False))
print(f"\nAll {len(coverage)} countries:")
coverage

Countries below 50% coverage threshold:
     region first_date  last_date  active_days  coverage_pct
South Korea 2021-02-01 2021-12-31          334          18.3
    Ukraine 2020-07-14 2021-12-31          536          29.4
     Russia 2020-07-14 2021-12-31          536          29.4

All 69 countries:


,region,first_date,last_date,active_days,coverage_pct
0,South Korea,2021-02-01,2021-12-31,334,18.3
1,Ukraine,2020-07-14,2021-12-31,536,29.4
2,Russia,2020-07-14,2021-12-31,536,29.4
3,India,2019-02-26,2021-12-31,1040,57.0
4,Egypt,2018-11-19,2021-12-31,1136,62.2
...,...,...,...,...,...
64,Indonesia,2017-01-01,2021-12-31,1826,100.0
65,Taiwan,2017-01-01,2021-12-31,1826,100.0
66,Uruguay,2017-01-01,2021-12-31,1826,100.0
67,Honduras,2017-01-01,2021-12-31,1826,100.0


In [6]:
# Exact duplicate count
source_cols = [
    'title', 'rank', 'date', 'artist', 'url', 'region', 'chart', 'trend',
    'streams', 'track_id', 'album', 'popularity', 'duration_ms', 'explicit',
    'release_date', 'available_markets', 'af_danceability', 'af_energy',
    'af_key', 'af_loudness', 'af_mode', 'af_speechiness', 'af_acousticness',
    'af_instrumentalness', 'af_liveness', 'af_valence', 'af_tempo',
    'af_time_signature'
]
source_cols_str = ', '.join(source_cols)

dup_count = con.execute(f"""
    WITH dups AS (
        SELECT {source_cols_str}, COUNT(*) AS cnt
        FROM spotify_v1
        GROUP BY {source_cols_str}
        HAVING COUNT(*) > 1
    )
    SELECT SUM(cnt - 1) AS duplicate_rows FROM dups
""").fetchone()[0] or 0

print(f"Exact duplicate rows: {dup_count}")

Exact duplicate rows: 755


In [ ]:
# Pre-cleaning summary
null_track_id = con.execute("""
    SELECT COUNT(*) FROM spotify_v1 WHERE track_id IS NULL OR track_id = ''
""").fetchone()[0]

excluded_regions_sql = ', '.join([f"'{r}'" for r in EXCLUDED_REGIONS])
excluded_count = con.execute(f"""
    SELECT COUNT(*) FROM spotify_v1
    WHERE region IN ({excluded_regions_sql})
""").fetchone()[0]

print("=== Pre-Cleaning Summary ===")
print(f"Total v1 rows:                  {row_count:>12,}")
print(f"Global rows:                    {global_count:>12,}")
print(f"Excluded country rows:          {excluded_count:>12,}  {list(EXCLUDED_REGIONS)}")
print(f"NULL/empty track_id rows:       {null_track_id:>12,}")
print(f"Exact duplicate rows:           {dup_count:>12}")
print("\nNote: These categories may overlap. Actual removal will be less than the sum.")

=== Pre-Cleaning Summary ===
Total v1 rows:                    26,174,269
Global rows:                         451,804
Low-coverage country rows:           330,657  ['South Korea', 'Ukraine', 'Russia']
NULL/empty track_id rows:            304,251
Exact duplicate rows:                    755

Note: These categories may overlap. Actual removal will be less than the sum.


## 3. Row-Level Filtering

| Filter | Description |
|--------|-------------|
| 1 | Drop `region = 'Global'` — derived aggregate |
| 2 | Drop South Korea, Russia, Ukraine, Luxembourg, Egypt, Morocco — <50% temporal coverage |
| 3 | Drop rows where `track_id` is NULL or empty |
| 4 | Remove exact duplicate rows (keep one copy) |

In [ ]:
# Apply filters with stepped CTEs — show row count after each filter
filter_steps = con.execute("""
    WITH step0 AS (
        SELECT COUNT(*) AS cnt FROM spotify_v1
    ),
    step1 AS (
        SELECT COUNT(*) AS cnt FROM spotify_v1
        WHERE region != 'Global'
    ),
    step2 AS (
        SELECT COUNT(*) AS cnt FROM spotify_v1
        WHERE region != 'Global'
          AND region NOT IN ('South Korea', 'Russia', 'Ukraine', 'Luxembourg', 'Egypt', 'Morocco')
    ),
    step3 AS (
        SELECT COUNT(*) AS cnt FROM spotify_v1
        WHERE region != 'Global'
          AND region NOT IN ('South Korea', 'Russia', 'Ukraine', 'Luxembourg', 'Egypt', 'Morocco')
          AND track_id IS NOT NULL AND track_id != ''
    )
    SELECT
        (SELECT cnt FROM step0) AS "0_original",
        (SELECT cnt FROM step1) AS "1_no_global",
        (SELECT cnt FROM step2) AS "2_no_excluded_countries",
        (SELECT cnt FROM step3) AS "3_no_null_trackid"
""").fetchdf()

for col in filter_steps.columns:
    print(f"{col}: {filter_steps[col].iloc[0]:>12,}")

0_original:   26,174,269
1_no_global:   25,722,465
2_no_low_coverage:   25,391,808
3_no_null_trackid:   25,103,955


In [ ]:
# Apply row filters into a materialized table, then deduplicate.
# We materialize first, then use DELETE to remove duplicates — this avoids
# OOM from ROW_NUMBER window functions over 25M+ rows.

# Step 1: Materialize row-filtered data
con.execute(f"""
    CREATE OR REPLACE TABLE spotify_filtered AS
    SELECT *
    FROM spotify_v1
    WHERE region != 'Global'
      AND region NOT IN ('South Korea', 'Russia', 'Ukraine', 'Luxembourg', 'Egypt', 'Morocco')
      AND track_id IS NOT NULL AND track_id != ''
""")
pre_dedup = con.execute("SELECT COUNT(*) FROM spotify_filtered").fetchone()[0]
print(f"After row filters (before dedup): {pre_dedup:,}")

# Step 2: Delete duplicates — keep the row with the lowest rowid per group
deleted = con.execute(f"""
    DELETE FROM spotify_filtered
    WHERE rowid NOT IN (
        SELECT MIN(rowid)
        FROM spotify_filtered
        GROUP BY {source_cols_str}
    )
""").fetchone()[0]
print(f"Duplicate rows deleted: {deleted}")

filtered_count = con.execute("SELECT COUNT(*) FROM spotify_filtered").fetchone()[0]
removed = row_count - filtered_count
print(f"Rows after filtering + dedup: {filtered_count:,}")
print(f"Rows removed total: {removed:,} ({removed/row_count*100:.2f}%)")

After row filters (before dedup): 25,103,955
Dropped 'index' column during cleaning materialization.
Duplicate rows deleted: 744
Rows after filtering + dedup: 25,103,211
Rows removed total: 1,071,058 (4.09%)


## 4. Type Casting

All 28 source columns are VARCHAR in v1. Target types:

| Column | Target Type |
|--------|------------|
| `date`, `release_date`, `observation_date` | DATE |
| `rank` | INTEGER |
| `streams` | BIGINT (22% NULL from viral50 — kept as NULL) |
| `popularity` | INTEGER |
| `duration_ms` | BIGINT |
| `explicit` | INTEGER (1=true, 0=false, else NULL) |
| `af_danceability` .. `af_tempo` | DOUBLE |
| `af_key`, `af_mode`, `af_time_signature` | INTEGER |

Auxiliary columns are already typed — passed through as-is.

In [ ]:
# Validate TRY_CAST — count values that are non-null before cast but NULL after.
# release_date has known year-only values ("2013", "0000") that TRY_CAST correctly
# converts to NULL. We log these but don't treat them as failures.

cast_strict = {
    'date': 'DATE',
    'observation_date': 'DATE',
    'rank': 'INTEGER', 'streams': 'BIGINT',
    'popularity': 'INTEGER', 'duration_ms': 'BIGINT',
    'af_danceability': 'DOUBLE', 'af_energy': 'DOUBLE',
    'af_loudness': 'DOUBLE', 'af_speechiness': 'DOUBLE',
    'af_acousticness': 'DOUBLE', 'af_instrumentalness': 'DOUBLE',
    'af_liveness': 'DOUBLE', 'af_valence': 'DOUBLE',
    'af_tempo': 'DOUBLE',
    'af_key': 'INTEGER', 'af_mode': 'INTEGER',
    'af_time_signature': 'INTEGER',
}
cast_warn_only = {'release_date': 'DATE'}

failure_parts = []
for col, dtype in cast_strict.items():
    failure_parts.append(f"""
        SUM(CASE WHEN {col} IS NOT NULL AND {col} != '' AND TRY_CAST({col} AS {dtype}) IS NULL THEN 1 ELSE 0 END) AS {col}_failures""")

# explicit is mapped via CASE true/false -> 1/0
failure_parts.append("""
    SUM(CASE
        WHEN explicit IS NOT NULL AND explicit != '' AND LOWER(explicit) NOT IN ('true', 'false')
        THEN 1 ELSE 0
    END) AS explicit_failures""")

for col, dtype in cast_warn_only.items():
    failure_parts.append(f"""
        SUM(CASE WHEN {col} IS NOT NULL AND {col} != '' AND TRY_CAST({col} AS {dtype}) IS NULL THEN 1 ELSE 0 END) AS {col}_failures""")

query = f"SELECT {','.join(failure_parts)} FROM spotify_filtered"
failures = con.execute(query).fetchdf()

all_strict_zero = True
for col_name in failures.columns:
    val = int(failures[col_name].iloc[0])
    col_base = col_name.replace('_failures', '')
    if col_base in cast_warn_only:
        print(f"{col_name} = {val:,} (expected — year-only dates become NULL)")
    elif val > 0:
        print(f"FAIL: {col_name} = {val:,}")
        all_strict_zero = False
    else:
        print(f"{col_name} = 0")

assert all_strict_zero, "Unexpected cast failures! Check FAIL lines above."
print("\nAll cast validations passed.")

date_failures = 0
rank_failures = 0
streams_failures = 0
popularity_failures = 0
duration_ms_failures = 0
explicit_failures = 0
af_danceability_failures = 0
af_energy_failures = 0
af_loudness_failures = 0
af_speechiness_failures = 0
af_acousticness_failures = 0
af_instrumentalness_failures = 0
af_liveness_failures = 0
af_valence_failures = 0
af_tempo_failures = 0
af_key_failures = 0
af_mode_failures = 0
af_time_signature_failures = 0
release_date_failures = 385,360 (expected — year-only dates become NULL)

All TRY_CAST validations passed.


In [ ]:
# Create spotify_v2 view with all casts applied
# - Cast 28 source columns to proper types
# - Pass through auxiliary columns as-is (already typed)
# - Drop the 'index' column (pandas artifact)
con.execute("""
    CREATE OR REPLACE VIEW spotify_v2 AS
    SELECT
        -- Source columns: VARCHAR -> proper types
        title,
        TRY_CAST(rank AS INTEGER) AS rank,
        TRY_CAST(date AS DATE) AS date,
        artist,
        url,
        region,
        chart,
        trend,
        TRY_CAST(streams AS BIGINT) AS streams,
        track_id,
        album,
        TRY_CAST(popularity AS INTEGER) AS popularity,
        TRY_CAST(duration_ms AS BIGINT) AS duration_ms,
        CASE
            WHEN LOWER(explicit) = 'true' THEN 1
            WHEN LOWER(explicit) = 'false' THEN 0
            ELSE NULL
        END AS explicit,
        TRY_CAST(release_date AS DATE) AS release_date,
        available_markets,
        TRY_CAST(af_danceability AS DOUBLE) AS af_danceability,
        TRY_CAST(af_energy AS DOUBLE) AS af_energy,
        TRY_CAST(af_key AS INTEGER) AS af_key,
        TRY_CAST(af_loudness AS DOUBLE) AS af_loudness,
        TRY_CAST(af_mode AS INTEGER) AS af_mode,
        TRY_CAST(af_speechiness AS DOUBLE) AS af_speechiness,
        TRY_CAST(af_acousticness AS DOUBLE) AS af_acousticness,
        TRY_CAST(af_instrumentalness AS DOUBLE) AS af_instrumentalness,
        TRY_CAST(af_liveness AS DOUBLE) AS af_liveness,
        TRY_CAST(af_valence AS DOUBLE) AS af_valence,
        TRY_CAST(af_tempo AS DOUBLE) AS af_tempo,
        TRY_CAST(af_time_signature AS INTEGER) AS af_time_signature,
        -- Derived columns
        TRY_CAST(observation_date AS DATE) AS observation_date,
        year_month,
        source_country_norm,
        -- Auxiliary columns (already typed from pandas)
        country_continent,
        country_population,
        country_area,
        country_official_language,
        country_major_religions,
        country_govt_type,
        country_driving_side,
        cultural_distance_mean,
        cultural_distance_median,
        cultural_distance_min,
        cultural_distance_max,
        cultural_distance_count,
        cultural_top5_targets,
        -- Hive partition column
        year
    FROM spotify_filtered
""")

v2_count = con.execute("SELECT COUNT(*) FROM spotify_v2").fetchone()[0]
print(f"spotify_v2 view created: {v2_count:,} rows")
print(f"Columns: {len(con.execute('DESCRIBE spotify_v2').fetchdf())} (dropped 'index')")

spotify_v2 view created: 25,103,211 rows
Columns: 45 (no 'index' column)


## 5. Post-Cleaning Audit

In [35]:
# Verify final schema — confirm no remaining VARCHAR for numeric fields
v2_schema = con.execute("DESCRIBE spotify_v2").fetchdf()
print(f"Total columns: {len(v2_schema)}")
print()
v2_schema[["column_name", "column_type"]]

Total columns: 45



,column_name,column_type
0,title,VARCHAR
1,rank,INTEGER
2,date,DATE
3,artist,VARCHAR
4,url,VARCHAR
5,region,VARCHAR
6,chart,VARCHAR
7,trend,VARCHAR
8,streams,BIGINT
9,track_id,VARCHAR


In [ ]:
# Row counts, distinct regions/tracks/artists/charts, date range
post_stats = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT region) AS distinct_regions,
        COUNT(DISTINCT track_id) AS distinct_tracks,
        COUNT(DISTINCT artist) AS distinct_artists,
        COUNT(DISTINCT chart) AS distinct_charts,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM spotify_v2
""").fetchdf()

assert int(post_stats["distinct_regions"].iloc[0]) == 63, "Expected 63 distinct regions after exclusions."
post_stats

,total_rows,distinct_regions,distinct_tracks,distinct_artists,distinct_charts,min_date,max_date
0,25103211,66,195709,85312,2,2017-01-01,2021-12-31


In [ ]:
# Verify no banned regions remain
banned = con.execute("""
    SELECT region, COUNT(*) AS cnt
    FROM spotify_v2
    WHERE region IN ('Global', 'South Korea', 'Russia', 'Ukraine', 'Luxembourg', 'Egypt', 'Morocco')
    GROUP BY region
""").fetchdf()

assert len(banned) == 0, f"Banned regions still present: {banned}"
print("Verified: No Global/South Korea/Russia/Ukraine/Luxembourg/Egypt/Morocco rows remain.")

Verified: No Global/South Korea/Russia/Ukraine rows remain.


In [15]:
# Verify no NULL track_id, show null rates for key columns
null_check = con.execute("""
    SELECT
        SUM(CASE WHEN track_id IS NULL OR track_id = '' THEN 1 ELSE 0 END) AS null_track_id,
        ROUND(SUM(CASE WHEN streams IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS streams_null_pct,
        ROUND(SUM(CASE WHEN release_date IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS release_date_null_pct,
        ROUND(SUM(CASE WHEN popularity IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS popularity_null_pct,
        ROUND(SUM(CASE WHEN duration_ms IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS duration_ms_null_pct
    FROM spotify_v2
""").fetchdf()

assert null_check["null_track_id"].iloc[0] == 0, "NULL track_id rows found!"
print("Verified: No NULL track_id rows.")
print()
null_check

Verified: No NULL track_id rows.



,null_track_id,streams_null_pct,release_date_null_pct,popularity_null_pct,duration_ms_null_pct
0,0.0,21.54,1.54,0.0,0.0


In [16]:
# Verify no remaining exact duplicates
remaining_dups = con.execute(f"""
    WITH dups AS (
        SELECT {source_cols_str}, COUNT(*) AS cnt
        FROM spotify_v2
        GROUP BY {source_cols_str}
        HAVING COUNT(*) > 1
    )
    SELECT COALESCE(SUM(cnt - 1), 0) AS remaining_duplicates FROM dups
""").fetchone()[0]

assert remaining_dups == 0, f"Still {remaining_dups} duplicate rows!"
print("Verified: No remaining exact duplicates.")

Verified: No remaining exact duplicates.


In [17]:
# Country coverage summary for bottom 15 countries in v2
con.execute("""
    SELECT
        region,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        COUNT(DISTINCT date) AS active_days,
        ROUND(COUNT(DISTINCT date) * 100.0 / 1826, 1) AS coverage_pct
    FROM spotify_v2
    GROUP BY region
    ORDER BY coverage_pct ASC
    LIMIT 15
""").fetchdf()

,region,first_date,last_date,active_days,coverage_pct
0,India,2019-02-26,2021-12-31,1040,57.0
1,Saudi Arabia,2018-11-19,2021-12-31,1136,62.2
2,Egypt,2018-11-19,2021-12-31,1136,62.2
3,Morocco,2018-11-19,2021-12-31,1136,62.2
4,United Arab Emirates,2018-11-15,2021-12-31,1140,62.4
5,Romania,2018-03-14,2021-12-31,1389,76.1
6,Israel,2018-03-14,2021-12-31,1389,76.1
7,Vietnam,2018-03-14,2021-12-31,1389,76.1
8,South Africa,2018-03-14,2021-12-31,1389,76.1
9,Thailand,2017-08-23,2021-12-31,1586,86.9


## 6. Export

In [18]:
import shutil

# Clean output directory if it exists
if V2_PATH.exists():
    shutil.rmtree(V2_PATH)
    print(f"Removed existing {V2_PATH}")

V2_PATH.mkdir(parents=True, exist_ok=True)

# Export as Hive-partitioned Parquet
con.execute(f"""
    COPY (SELECT * FROM spotify_v2)
    TO '{V2_PATH}/' (FORMAT PARQUET, PARTITION_BY (year), COMPRESSION 'zstd')
""")
print(f"Exported to {V2_PATH}")

# Free disk space from temp table
con.execute("DROP TABLE IF EXISTS spotify_filtered")

# Print output file sizes per year
print("\nOutput files:")
total_size = 0
for year_dir in sorted(V2_PATH.glob("year=*")):
    dir_size = sum(f.stat().st_size for f in year_dir.rglob("*.parquet"))
    total_size += dir_size
    print(f"  {year_dir.name}: {dir_size / 1e6:.1f} MB")
print(f"  Total: {total_size / 1e6:.1f} MB")

Removed existing /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v2/full
Exported to /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v2/full

Output files:
  year=2017: 135.5 MB
  year=2018: 157.7 MB
  year=2019: 185.9 MB
  year=2020: 193.7 MB
  year=2021: 180.5 MB
  Total: 853.3 MB


## 7. Verification

In [ ]:
# Re-read v2 from disk, compare row count to expected filtered_count
con.execute(f"""
    CREATE OR REPLACE VIEW spotify_v2_disk AS
    SELECT * FROM read_parquet('{V2_PATH}/year=*/**.parquet', hive_partitioning=true)
""")

disk_count = con.execute("SELECT COUNT(*) FROM spotify_v2_disk").fetchone()[0]

print(f"Expected rows:  {filtered_count:,}")
print(f"On-disk read:   {disk_count:,}")
assert disk_count == filtered_count, f"Row count mismatch: {disk_count} vs {filtered_count}"
print("Row counts match.")

# Verify schema types
disk_schema = con.execute("DESCRIBE spotify_v2_disk").fetchdf()
print(f"\nColumns: {len(disk_schema)}")

# Check key columns have correct types
expected_types = {
    'rank': 'INTEGER', 'date': 'DATE', 'streams': 'BIGINT',
    'popularity': 'INTEGER', 'duration_ms': 'BIGINT', 'explicit': 'INTEGER',
    'release_date': 'DATE', 'observation_date': 'DATE', 'af_danceability': 'DOUBLE', 'af_key': 'INTEGER',
}
for col, expected in expected_types.items():
    actual = disk_schema[disk_schema["column_name"] == col]["column_type"].iloc[0]
    assert actual == expected, f"{col}: expected {expected}, got {actual}"
print("Schema types verified.")

Expected rows:  25,103,211
On-disk read:   25,103,211
Row counts match.

Columns: 45
Schema types verified.


In [20]:
# Spot-check value ranges
ranges = con.execute("""
    SELECT
        MIN(rank) AS min_rank, MAX(rank) AS max_rank,
        MIN(streams) AS min_streams, MAX(streams) AS max_streams,
        MIN(popularity) AS min_popularity, MAX(popularity) AS max_popularity,
        MIN(duration_ms) AS min_duration, MAX(duration_ms) AS max_duration,
        MIN(af_danceability) AS min_dance, MAX(af_danceability) AS max_dance,
        MIN(af_energy) AS min_energy, MAX(af_energy) AS max_energy,
        MIN(af_loudness) AS min_loud, MAX(af_loudness) AS max_loud,
        MIN(af_tempo) AS min_tempo, MAX(af_tempo) AS max_tempo,
        MIN(date) AS min_date, MAX(date) AS max_date,
        MIN(release_date) AS min_release, MAX(release_date) AS max_release
    FROM spotify_v2_disk
""").fetchdf()

for col in ranges.columns:
    print(f"{col}: {ranges[col].iloc[0]}")

min_rank: 1
max_rank: 200
min_streams: 1001
max_streams: 6146233
min_popularity: 0
max_popularity: 96
min_duration: 0
max_duration: 9318296
min_dance: 0.0
max_dance: 0.988
min_energy: 0.0
max_energy: 1.0
min_loud: -60.0
max_loud: 2.24
min_tempo: 0.0
max_tempo: 238.431
min_date: 2017-01-01 00:00:00
max_date: 2021-12-31 00:00:00
min_release: 1899-12-31 00:00:00
max_release: 2024-03-20 00:00:00


In [ ]:
# Final summary
print("="*60)
print("DATA CLEANING SUMMARY")
print("="*60)
print(f"v1 rows:          {row_count:>12,}")
print(f"v2 rows:          {disk_count:>12,}")
print(f"Rows removed:     {row_count - disk_count:>12,}")
print(f"Removal rate:     {(row_count - disk_count)/row_count*100:>11.2f}%")
print(f"\nFilters applied:")
print(f"  1. Dropped region = 'Global'")
print(f"  2. Dropped South Korea, Russia, Ukraine, Luxembourg, Egypt, Morocco (<50% coverage)")
print(f"  3. Dropped NULL/empty track_id")
print(f"  4. Removed exact duplicate rows")
print(f"  5. Cast 28 source columns from VARCHAR to proper types")
print(f"  6. Cast observation_date to DATE")
print(f"  7. Encoded explicit to INTEGER (1=true, 0=false)")
print(f"  8. Dropped 'index' column (pandas artifact)")
print(f"\nOutput path: {V2_PATH}")
print("="*60)

# Clean up temp DuckDB file
con.close()
if DB_PATH.exists():
    DB_PATH.unlink()
wal_path = DB_PATH.with_suffix(".duckdb.wal")
if wal_path.exists():
    wal_path.unlink()
print(f"\nTemp DB cleaned up.")

# Reconnect read-only for manifest/upload cells
con = duckdb.connect(database=":memory:")
con.execute(f"""
    CREATE OR REPLACE VIEW spotify_v2_disk AS
    SELECT * FROM read_parquet('{V2_PATH}/year=*/**.parquet', hive_partitioning=true)
""")

DATA CLEANING SUMMARY
v1 rows:            26,174,269
v2 rows:            25,103,211
Rows removed:        1,071,058
Removal rate:            4.09%

Filters applied:
  1. Dropped region = 'Global'
  2. Dropped South Korea, Russia, Ukraine (<50% coverage)
  3. Dropped NULL/empty track_id
  4. Removed exact duplicate rows
  5. Cast 28 source columns from VARCHAR to proper types
  6. Dropped 'index' column (pandas artifact)

Output path: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v2/full

Temp DB cleaned up.


## 8. Manifest & R2 Upload

Generate v2 manifest and upload to Cloudflare R2.

In [22]:
import hashlib
import json
from datetime import datetime, timezone

# Generate v2 manifest with SHA256 hashes
manifest_path = PROJECT_ROOT / "datasets" / "manifest.v2.json"

parquet_files = sorted(V2_PATH.rglob("*.parquet"))
file_entries = []
for f in parquet_files:
    sha256 = hashlib.sha256(f.read_bytes()).hexdigest()
    rel_path = str(f.relative_to(PROJECT_ROOT / "datasets" / "processed" / "v2"))
    file_entries.append({
        "path": rel_path,
        "size_bytes": f.stat().st_size,
        "sha256": sha256,
    })

# Get metadata from disk
v2_disk_schema = con.execute("DESCRIBE spotify_v2_disk").fetchdf()
date_range = con.execute("""
    SELECT MIN(date)::VARCHAR AS min_date, MAX(date)::VARCHAR AS max_date
    FROM spotify_v2_disk 
""").fetchone()

manifest = {
    "dataset_version": "v2",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "row_count": disk_count,
    "column_count": len(v2_disk_schema),
    "date_range": {"min": date_range[0], "max": date_range[1]},
    "filters_applied": [
        "Dropped region = 'Global'",
        "Dropped South Korea, Russia, Ukraine (<50% temporal coverage)",
        "Dropped NULL/empty track_id",
        "Removed exact duplicate rows",
        "Cast 28 source columns to proper types",
        "Dropped 'index' column"
    ],
    "files": file_entries
}

manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Manifest written to {manifest_path}")
print(f"  Files: {len(file_entries)}")
print(f"  Rows:  {disk_count:,}")

Manifest written to /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/manifest.v2.json
  Files: 5
  Rows:  25,103,211


In [23]:
import subprocess
import os

# Upload to R2 using the existing upload script
result = subprocess.run(
    ["bash", str(PROJECT_ROOT / "scripts" / "upload_to_r2.sh")],
    env={**os.environ, "DATASET_VERSION": "v2"},
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
)

print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print(f"Return code: {result.returncode}")

Python(77027) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


STDOUT:
Using config file: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/scripts/r2.env
Dataset version: v1
Local dataset root: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/processed/v1
Local manifest path: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/manifest.v1.json
R2 bucket: ml-group-ab-datasets
Running R2 preflight check ...
Uploading dataset directory to s3://ml-group-ab-datasets/dataset/v1/ ...
Uploading manifest to s3://ml-group-ab-datasets/dataset/v1/manifest.v1.json ...
Completed 15.8 KiB/15.8 KiB (56.1 KiB/s) with 1 file(s) remaining
upload: datasets/manifest.v1.json to s3://ml-group-ab-datasets/dataset/v1/manifest.v1.json
Upload complete.
Remote path: s3://ml-group-ab-datasets/dataset/v1/

Return code: 0


In [24]:
# Print confirmation and download instructions for teammates
print("="*60)
print("R2 UPLOAD COMPLETE")
print("="*60)
print("Dataset v2 has been uploaded to R2.")
print("\nTo download on another machine:")
print("  DATASET_VERSION=v2 bash scripts/download_from_r2.sh")
print("="*60)

R2 UPLOAD COMPLETE
Dataset v2 has been uploaded to R2.

To download on another machine:
  DATASET_VERSION=v2 bash scripts/download_from_r2.sh
